# CipherCore

## Fundamentos Matemáticos e Computacionais para Triagem de Ameaças (SecOps)

### 1. Abstração e objetivo do sistema
Este documento estabelece a nossa base teórica para o **CipherCore**, um terminal interativo projetado para analistas de segurança da informação. O objetivo deste sistema não é substituir ferramentas consolidadas de mercado, mas simular a fundação de **Detecção de Anomalias** e **Análise de Artefatos** utilizando primitivas matemáticas e controle rigoroso de fluxo computacional.

Na cibersegurança, malwares e algoritmos de ofuscação exploram os limites da arquitetura de computadores. Como analistas, precisamos reverter esse processo. Para isso, modelamos a triagem de ameaças como um sistema de funções discretas, onde um conjunto de dados suspeitos $X$ é submetido a uma função de inspeção $f(x)$, resultando em uma classificação de risco $Y$:

$$ f: X \rightarrow Y $$

lê-se: **A função $f$ mapeia o domínio $X$ no contradomínio $Y$."**

* **$f$ (Função)**: é a regra da transformação, o mecanismo ou o algoritmo em si.
* **$X$ (Domínio)**: é o conjunto de todas as entradas (*inptus*) estritamente válidas. No *CipherCore*, isso representa o conjunto de todos os *payloads* de rede interceptados.
* **$Y$ (Contradomínio)**: é o conjunto de todas as saídas (*outputs*) estruturalmente possíveis. No nosso cenário seriam as classificações (ex: "benigno" ou "anômalo").

- Exemplo na matemática pura
Vamos definir uma função elementar que eleva um número ao quadrado.
Seja o domínio $X$ o conjunto estrito de inteiros $\{1, 2, 3\}$.
A regra de transformação é dada por:

$$f(x) = x^2$$

O mapeamento desta função resultará no contradomínio $Y = \{1, 4, 9\}$, através das seguintes relações discretas:

* $f(1) = 1$
* $f(2) = 4$
* $f(3) = 9$

Onde o domínio $X$ representa logs, chaves ou payloads, e o contradomínio $Y$ representa o estado da ameaça (ex: benigno, anômalo, malicioso).

Na engenharia de software moderna, representamos esse rigor matemático de domínio e contradomínio utilizando a tipagem de dados (*Type Hiting*):

In [ ]:
def f(x: int) -> int:
    """
    Função matemática elementar.
    Mapeia um número inteiro (Domínio X) para o seu quadrado (Contradomínio Y).
    """
    y = x ** 2
    return y

### 2. Pilares Arquiteturais do Toolkit

O sistema que vamos construir traduz conceitos matemáticos puros para o monitoramento de segurança através de quatro eixos principais:

#### 2.1. Criptoanálise e Teoria dos Números
A criptografia moderna baseia-se na assimetria computacional da fatoração de inteiros. Nosso módulo matemático aplicará testes de primalidade e busca por aproximações de raízes para simular falhas em geração de chaves. Se $N$ é uma chave pública, a segurança do sistema depende de $N = p \times q$, onde $p$ e $q$ são números primos suficientemente grandes. Nosso sistema aplicará força bruta otimizada para compreender o colapso estrutural de chaves fracas.

#### 2.2. Simetria e Sanitização de Strings (Ofuscação)
Atacantes tentam evadir assinaturas estáticas manipulando a representação de caracteres. Definimos uma string $S$ de comprimento $n$. O sistema identificará anomalias estruturais, como espelhamento de *payloads*, verificando a condição rigorosa de palíndromos através da iteração de índices espaciais, respeitando a condição matemática:

$$ \forall i \in \{0, 1, ..., \lfloor n/2 \rfloor - 1\}, \ S[i]\equiv S[n - 1 - i] $$

lê-se: para todo índice $i$ pertencente ao conjunto dos números inteiros que vai de $0$ até o piso da metade do comprimento da string *menos um*, o caractere na posição $i$ deve ser estritamente equivalente ao caractere na posição oposta reflexiva $n - 1 - 1$.

* **$\forall$ (para todo):** Indica um laço de repetição. A condição não pode falhar nenhuma vez. Se falhar, a string não é um palíndromo.
* **$\lfloor n/2 \rfloor$ (piso da divisão):** Garante que, em strings de tamanho ímpar, o elemento central (índice 2) não seja comparado com ele mesmo, evitando ciclo de desperdício de CPU.
* **$\equiv$ (equivalência):** A comparação de identidade entre os dois polos da string.

- Exemplo
Seja a string $S = \text{"RADAR"}$. O seu comprimento é $n = 5$.
O limite de nosso conjunto de índices é calculado como: $\lfloor 5/2 \rfloor - 1 = 2 - 1 = 1$.
Portanto, o conjunto de índices $i$ que precisamos testar é $\{0, 1\}$.

Executando passo a passo:

* **Para $i = 0$:**
$S[0] \equiv S[5 - 1 - 0] \rightarrow S[0] \equiv S[4] \rightarrow \text{'R'}$ (Verdadeiro)

* **Para $i = 1$:**
$S[1] \equiv S[5 - 1- 1] \rightarrow S[1] \equiv S[3] \rightarrow \text{'A'} \equiv \text{'A'}$ (Verdadeiro)

Como a proposição se manteve verdadeira para todo $i$ dentro do conjunto delimitado, $S$ é declarada um palíndromo.

Em Python, traduz-se do seguinte modo a condição acima:

In [ ]:
def verifica_simetria_matematica(s: str) -> bool:
    """
    Motor matemático puro para verificação de palíndromos.
    Não inclui sanitização de dados.

    Args:
    - s (str): a string a ser testada se é um palíndromo.

    Returns:
    - bool: `True` se `s` for um palíndromo, `False`, caso contrário.
    """
    n = len(s)

    # range(k) cria o conjunto {0, 1, ..., k - 1}
    # range(n // 2) traduz o conjunto matemático {0, 1, ..., floor(n/2) - 1}
    conjunto_indices = range(n // 2)

    for i in conjunto_indices:
        # A condicional inverte a lógica do $\forall$ (*Fail-Fast*).
        # Se achar uma única falha, quebra a prova e retorna Falso
        # imediatamente.
        if s[i] != s[n - 1 - i]:
            return False

    return True

A operação de sanitização prévia garante que os elementos de $S$ sejam avaliados independentemente de sua entropia visual (letras maiúsculas ou minúsculas).

#### 2.3. Emulação Primitiva (Análise de Shellcode)
Para contornar heurísticas, malwares de baixo nível reduzem operações complexas à aritmética elementar da Unidade Lógica Aritmética (ALU). Nosso sistema modelará algoritmos de divisão e multiplicação baseados puramente em ciclos de subtração e soma:

$$ a \times b = \sum_{k=1}^{b} a \quad \text{e} \quad \frac{a}{b} = \max \left\{ n \in \mathbb{N} \mid a - n \cdot b \geq 0 \right\} $$

lê-se: "O produto de $a$ e $b$ é igual ao somatório da constante $a$, onde o índice $k$ varia de $1$ até $b$".
Se o nosso *shellcode* precisar alocar $4$ blocos de $3$ bytes, o cálculo de $4 \times 3$ é resolvido pelo processador primitivo como:
$$ 4 \times 3 = \sum_{k=1}^{3} 4 = 4 + 4 + 4 = 12 $$

A divisão Euclidiana restrita aos números inteiros é formalizada pela busca do maior multiplicador possível antes que o resultado se torne negativo:
$$ \frac{a}{b} = \max \{n \in \mathbb{N} \mid a - n \cdot b \geq 0 \} $$
lê-se: O quociente da divisão de $a$ por $b$ é o valor máximo contido no conjunto dos números naturais $n$, tal que a diferença entre $a$ e o produto de $n$ por $b$ seja maior ou igual a zero.

Se o *shellcode* precisa fatiar um buffer de $13$ bytes em pedaços de $4$ bytes ($13 / 4$):
Testamos os naturais $n = \{0, 1, 2, 3, 4\}$:
* $13 - (0 \cdot 4) = 13 \geq 0$
* $13 - (1 \cdot 4) = 9 \geq 0$
* $13 - (2 \cdot 4) = 5 \geq 0$
* $13 - (3 \cdot 4) = 1 \geq 0$
* $13 - (4 \cdot 4) = -3$ (falha, pois $-3 \not\geq 0$)

O valor máximo de $n$ que satisfaz a condição é $3$. O resto (módulo) é o resíduo da última operação válida, que é $1$.

As duas operações em Python podem ser representadas pelas funções:

In [1]:
def multiplicacao_primitiva(a: int, b: int) -> int:
    """
    Emula a multiplicação através do somatório discreto.
    """
    acumulador = 0
    # O laço for simula o índice k variando de 1 até b
    for _ in range(b):
        acumulador += a
    return acumulador

def divisao_inteira_primitiva(a: int, b: int) -> tuple[int, int]:
    """
    Emula a divisão buscando o máximo de subtrações sucessivas.

    Returns:
    - tuple: quociente, resto
    """
    if b == 0:
        raise ZeroDivisionError("Divisão por zero não pertence aos reais.")

    n = 0
    resto = a

    # Simula a condição (a - n*b >= 0)
    while resto - b >= 0:
        resto -= b
        n += 1

    return n, resto

Isso treina o analista a ler *assembly* malicioso e a entender limites de registradores.

#### 2.4. Modelagem de Impacto (Séries Discretas)
A propagação de *ramsomwares* e a exfiltração de dados seguem modelos de progressão acumulativa. Utilizaremos matemática financeira (juros compostos e progressões) e algoritmos gulosos para simular a fragmentação de pacotes de dados ou a escalada de prejuízos ao longo do tempo $t$, calculando o montante $M$:

$$ M(t) = C \cdot (1 + i)^t $$

lê-se: O montante do impacto $M$ em função do tempo $t$ é igual ao custo inicial $C$, multiplicado pelo fator de acumulação composto de $um$, mais a taxa de degradação $i$, elevado à potência do tempo $t$.

* **$M(t)$ (Montante do tempo $t$):** a variável dependente. O prejuízo total ou volume de dados comprometidos no instante $t$.
* **$C$ (Capital/Custo Base):** o impacto inicial no momento zero (ex: valor da primeira máquina comprometida).
* **$i$ (Taxa de degradação):** a velocidade de propagação ou o juro cobrado pelo atacante (em formato decimal).
* **$t$ (tempo):** o número de ciclos discretos (dias, horas) em que o incidete permanece sem mitigação.

Um servidor é comprometido com custo de inatividade estimado em $C = \text{R\$} \ 1.000,00$ no dia 0. O ataque se espalha pela infraestrutura aumentando o dano a uma taxa de $10\%$ ao dia ($i = 0,10$). Queremos saber o impacto no 3º dia ($t = 3$).

Aplicando a equação:
$$ M(3) = 1000 \cdot (1 + 0,10)^3 $$
$$ M(3) = 1000 \cdot (1,10)^3 $$
$$ M(3) = 1000 \cdot 1,331 $$
$$ M(3) = 1331 $$

O impacto no terceiro dia é de $\text{R\$}\ 1.331,00$

Em Python temos:

In [2]:
def simula_impacto_exponencial(
        custo_inicial: float,
        taxa_degradacao: float,
        tempo: int,
        ) -> float:
    """
    Calcula o impacto financeiro ou estrutural cumulativo de um incidente de segurança.

    Args:
    - custo inicial (float): valor do prejuízo causado pelo incidente no marco 0.
    - taxa_degradacao (float): taxa de degradação expressa em formato decimal.
    - tempo (int): tempo usado para cálculo do montante do prejuízo.

    Returns:
    - float: montante do prejuízo no intervalo de tempo indicado.
    """
    impacto_total = custo_inicial * ((1 + taxa_degradacao) ** tempo)
    return impacto_total

#### 2.5. Otimização Algorítmica: O Limite da Primalidade

Na Teoria dos Números e na Criptografia (como na geração de chaves RSA), a verificação de primalidade por força bruta de um número $N$ possui complexidade de tempo $O(N)$, o que é computacionalmente inviável para valores elevados. No entanto, o teorema dos fatores determina que é matematicamente impossível que o menor divisor próprio de um número composto seja maior que a sua raiz quadrada.

O limite superior rigoroso que o nosso laço de repetição deve atingir é definido pela fronteira:
$$ \text{Fronteira} =\lfloor \sqrt{N} \rfloor $$

##### 2.5.1. A Prova Lógica (por Contradição)

Seja $N$ um número inteiro composto. Por definição, ele pode ser fatorado no produto de dois números inteiros $a$ e $b$, onde ambos são maiores que $1$:
$$ N = a \cdot b $$
Assuma, por hipótese temporária, que ambos os fatores $a$ e $b$ sejam estritamente maiores que a raiz quadrada de $N$:
$$ a > \sqrt{N} \quad \text{e} \quad b > \sqrt{N} $$
Se multiplicarmos essas duas desigualdades, obtemos um colapso lógico:
$$ a \cdot b > \sqrt{N} \cdot \sqrt{N} $$
$$ a \cdot b > N $$
Como a nossa premissa inicial define que $N = a \cdot b$, a afirmação matemática de que $a \cdot b > \cdot b > N$ é uma contradição absoluta.
**Conclusão Estrutural:**
É impossível que ambos os fatores de um número sejam maiores que $\sqrt{N}$. Obrigatoriamente, pelo menos um dos fatores (por exemplo, $a$) deve ser menor ou igual à raiz quadrada de $N$:
$$ a \le \sqrt{N} $$
**Aplicação no código:**
Se o laço iterativo do algoritmo de segurança testar os divisores a partir de $3$ e atingir o piso de $\sqrt{N}$ sem encontrar nenhuma divisão com resto igual a zero, a prova está matematicamente exaurida. O ciclo de processamento deve ser interrompido imediatamente (*break*), pois o número $N$ é irrevogavelmente primo.

In [3]:
def verifica_primalidade(n: int) -> bool:
    """
    Verifica se um número é primo.

    Args:
    - n (int): o número inteiro a ser testado.

    Returns:
    - bool: `True` se for primo, `False`, caso contrário.
    """
    if n <= 1:
        return False

    elif n == 2:
        return True

    elif n % 2 == 0:
        return False

    limite = int(n ** 0.5)
    divisor = 3

    while divisor <= limite:
        if n % divisor == 0:
            return False
        divisor += 2

    return True

#### 2.6. Busca Assintótica: Intervalos e o Postulado de Bertrand

A busca autônoma pelo próximo número primo a partir de um limite arbitrário $N$ não é uma varredura cega; ela é amparada por dois pilares fundamentais da Teoria dos Números que garantem a completude e a eficiência temporal do algoritmo.

##### 2.6.1. O Postulado de Bertrand (Teorema de Chebyshev)

A primeira objeção estrutural a um algoritmo de busca indeterminada seria o risco do *infinite loop*: "E se o próximo primo estiver a uma distância computacionalmente inalcançável, exaurindo a memória do sistema?". O Postulado de Bertrand neutraliza essa falha ao provar rigorosamente que, para qualquer número inteiro $N > 1$, existe sempre pelo menos um número primo $p$ contido no intervalo:
$$ N < p < 2N $$
Isso estabelece uma fronteira superior matemática inquebrável. O algoritmo jamais varrerá o vazio rumo ao infinito; a distância máxima até o sucesso é limitada ao próprio valor de $N$.

##### 2.6.2. O Teorema do Número Primo (PNT)

Enquanto o postulado de Bertrand garante a existência teórica, o Teorema do Número Primo (Prime Number Theorem) define o custo operacional da arquitetura. A distribuição dos primos na reta numérica torna-se mais esparsa à medida que os números crescem. A distância média entre um inteiro arbitrário $N$ e o próximo número primo é logarítmica, aproximada pela equação:
$$ \text{Gap} \approx \ln(N) $$
Essa premissa prova que uma iteração sequencial de força bruta com salto unitário (testar $N+1$, $N+2$, $N+3$ \dots$) é estatisticamente supereficiente. Para um valor de magnitude de $10^9$ (um bilhão), a máquina de estados testará, em média, apenas $\approx 21$ candidatos antes de cravar uma colisão prima.

In [4]:
def gera_proximo_primo(n: int) -> int:
    """
    Gera o próximo número primo, após fornecido um número inteiro.

    Args:
    - n (int): o inteiro que precede o próximo número primo.

    Returns:
    - int: o número primo seguinte ao encontrado no domínio.
    """
    candidato = n + 1

    while True:
        if verifica_primalidade(candidato):
            return candidato

        candidato += 1

#### 2.7. O Algoritmo de Euclides (Máximo Divisor Comum)

Na engenharia criptográfica, provar que dois números inteiros são coprimos exige o cálculo do Máximo Divisor Comum (MDC). O algoritmo de Euclides elimina a necessidade de fatoração por força bruta utilizando o princípio da divisão com resto de forma recursiva.

O teorema fundamental estabelece que o MDC de dois inteiros $A$ e $B$ é matematicamente idêntico ao MDC de $B$ e o resto da divisão de $A$ por $B$. Em notação modular:
$$ \text{MDC}(A, B) = \text{MDC}(B, A \pmod B) $$

A otimização algorítmica reside na substituição sucessiva: o divisor anterior torna-se o novo dividendo, e o resto anterior torna-se o novo divisor. Esse ciclo de reduções simultâneas continua até que o resto atinja o valor exato de zero.
$$ R_n = 0 $$

No milissegundo em que o resto zera, a operação alcança a sua singularidade. O divisor remanescente nessa última equação válida não pode mais ser reduzido e é, incontestavelmente, o Máximo Divisor Comum procurado.

In [5]:
def calcula_mdc(a: int, b: int) -> int:
    """
    Calcula o máximo divisor comum de dois inteiros usando o
    algoritmo de Euclides.

    Args:
    - a (int): o primeiro inteiro do domínio.
    - b (int): o segundo inteiro do domínio.

    Returns:
    - int: o máximo divisor comum de `a` e `b`.
    """
    while b != 0:
        a, b = b, a % b

    return a

#### 2.8. O Expoente Público RSA e a Função Totiente

Na arquitetura do algoritmo RSA, a chave pública não é um único número, mas sim um par de valores: o Módulo Numérico ($N$) e o Expoente Público ($e$).

Para que a criptografia funcione de forma reversível e segura, o expoente $e$ não pode ser escolhido aleatoriamente. Ele deve obedecer a uma restrição matemática estrita em relação à Função Totiente de Euler ($\phi$), que representa a quantidade de números menores que $N$ que são coprimos com $N$.

As regras absolutas para a seleção do Expoente Público são:
1. $1 < e < \phi$
2. $\text{MDC}(e, \phi) = 1$ (Ou seja, $e$ e $\phi$ devem ser obrigatoriamente coprimos).

Na prática de engenharia de software, para otimizar o tempo de processamento, não sorteamos valores randômicos para $e$. Nós partimos do menor limite inferior válido e iteramos de forma sequencial até encontrar a primeira colisão matemática que satisfaça a condição de comprimalidade com $\phi$.

In [ ]:
def gera_expoente_publico(phi: int) -> int:
    """
    Gera uma chave pública RSA a partir da coprimalidade de um
    expoente e phi.

    Args:
    - phi (int): número que servirá como determinação da coprimalidade
    em relação ao expoente público.

    Returns:
    - int: o expoente público quando encontrada a coprimalidade.
    """
    e = 2
    while calcula_mdc(e, phi) != 1:
        e += 1

    return e

#### 2.9. O Inverso Multiplicativo Modular (Chave Privada RSA)

A segurança estrutural da criptografia RSA baseia-se na assimetria das chaves. Enquanto a chave pública ($e$) é distribuída abertamente para encriptar os dados, a chave privada ($d$) deve ser mantida em sigilo absoluto para decriptá-los.

Matematicamente, $d$ é definido como o inverso multiplicativo modular de $e$ em relação ao módulo $\phi$. Isso significa que a multiplicação da chave privada pela chave pública, quando dividida por $\phi$, deve deixar um resto rigorosamente igual a 1. A equação fundamental é:
$$ (d \cdot e) \pmod \phi = 1 $$

Encontrar $d$ por força bruta (testando $1, 2, 3 \dots$) é computacionalmente impossível para o escopo de chaves de 2048 bits. A solução assintótica exige o **Algoritmo de Euclides Estendido**. Ao invés de apenas fatiar os restos até zero, o algoritmo rastreia os coeficientes da Identidade de Bézout de forma reversa, reconstruindo a equação para encontrar o multiplicador exato em tempo logarítmico $O(\log \phi)$.

In [2]:
def calcula_inverso_modular(e: int, phi: int) -> int:
    """
    Calcula o inverso multiplicativo modular do expoente público
    em relação ao módulo de phi.

    Args:
    - e (int): o expoente público.
    - phi (int): função totiente de Euler.

    Returns:
    - int: o inverso multiplicativo postivo.
    """
    t, novo_t = 0, 1
    r, novo_r = phi, e

    while novo_r != 0:
        quociente = r // novo_r
        t, novo_t = novo_t, t - quociente * novo_t
        r, novo_r = novo_r, r - quociente * novo_r

    return t % phi

#### 2.10. Exponenciação Modular Rápida (Algoritmo Square-and-Multiply)

Na criptografia RSA, a encriptação e decriptação exigem a elevação de uma mensagem base (base $M$) a um expoente colossal (chave $E$ ou $D$), seguido da extração do resto pelo módulo numérico $N$:

$$ C = M^E \pmod N $$

Computar $M^E$ em seu valor absoluto antes de aplicar o módulo exige uma quantidade de memória e ciclos de processamento impraticáveis para chaves criptográficas modernas (2048 bits ou superiores). A solução algorítmica é a **Exponenciação Modular Rápida**. Este método fragmenta a operação utilizando a representação binária do expoente e a propriedade distributiva da aritmética modular:

$$ (A \cdot B) \pmod N = [(A \pmod N) \cdot(B \pmod N)] \pmod N $$

A arquitetura reduz a complexidade temporal de $O(E)$ para $O(\log E)$. O estado inicial exige que a base sofra uma redução modular preventiva ($M \pmod N$). Em cada ciclo iterativo, o algoritmo avalia a paridade do bit menos significativo do expoente. Se ímpar, o acumulador multiplica a base atual e extrai o módulo. Independentemente da paridade, a base é elevada ao quadrado e reduzida, e o expoente sofre um deslocamento de bit à direita (divisão inteira por 2) até ser aniquilado a zero.



In [ ]:
def exponenciacao_modular(base: int, expoente: int, modulo: int) -> int:
    """
    Calcula a exponenciação modular utilizando a representação binária
    do expoente e a propriedade distributiva da aritmética módular.

    Args:
    - base (int): mensagem base.
    - expoente (int): expoente colossal.
    - modulo (int): o módulo numérico aplicado.

    Returns:
    - int: a chave de decriptação.
    """
    resultado = 1
    base = base % modulo

    while expoente > 0:
        if expoente % 2 == 1:
            resultado = (resultado * base) % modulo

        base = (base ** 2) % modulo
        expoente = expoente // 2

    return resultado

: 

### 3. Orquestração de Chaves RSA

A segurança do algoritmo RSA não reside apenas em suas operações matemáticas isoladas, mas na sequência inquebrável da sua arquitetura. O orquestrador de chaves elimina a intervenção humana transitória, garantindo que o módulo numérico ($N$) e a função totiente ($\phi$) sejam derivados estritamente de números primos absolutos e validados preventivamente.

As chaves são estruturalmente compostas por tuplas duplas:
- Chave Pública: $(e, N)$ - Distribuída em canais abertos para encriptação.
- Chave Privada: $(d, N)$ - Retida em absoluto sigilo para decriptação.

O motor de orquestração assenta-se sobre duas funções geométricas:
1. $N = P \cdot Q$
2. $\phi = (P -1) \cdot (Q - 1)$

Com essas variáveis alocadas, o expoente público ($e$) e seu respectivo inverso modular ($d$) são derivados, e o par criptográfico é formalmente instanciado no sistema.

---

**Diretriz de Execução:** A partir deste modelo matemático, cada módulo será implementado isoladamente no código-fonte, garantindo baixo acoplamento e alta coesão, sem a utilização de bibliotecas que mascarem a lógica de controle de estado.